In [ ]:
pip install yfinance

In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
from datetime import datetime
import statsmodels.api as sm



def validar_y_convertir_fechas(fechas):
    """
    Convierte una lista de fechas a formato 'YYYY-MM-DD' si no están ya en ese formato.

    Parámetros:
    - fechas: Lista de fechas como cadenas.

    Retorna:
    - Lista de fechas en formato 'YYYY-MM-DD'.
    """
    fechas_convertidas = []
    for fecha in fechas:
        try:
            # Intenta parsear la fecha asumiendo el formato 'YYYY-MM-DD'
            fecha_convertida = datetime.strptime(fecha, '%Y-%m-%d')
        except ValueError:
            # Intenta otros formatos si el anterior falla
            fecha_convertida = pd.to_datetime(fecha, errors='coerce')
            if pd.isnull(fecha_convertida):
                raise ValueError(f"Fecha no reconocida: {fecha}")
        # Asegura el formato correcto
        fechas_convertidas.append(fecha_convertida.strftime('%Y-%m-%d'))
    return fechas_convertidas

def obtener_precios_logaritmicos(ticker, fechas):
    """
    Descarga los datos de precios de un ticker específico y calcula la variación logarítmica
    de los precios de cierre para las fechas dadas, y luego formatea los resultados en porcentaje.

    Parámetros:
    - ticker: El símbolo del ticker de la acción (como string).
    - fechas: Lista de fechas en formato 'YYYY-MM-DD'.

    Retorna:
    - Un DataFrame con las fechas dadas y las variaciones logarítmicas de los precios de cierre entre ellas,
      formateadas en porcentaje con el símbolo '%'.
    """
    # Descargar datos de un rango que cubra las fechas dadas
    datos = yf.download(ticker, start=min(fechas), end=max(fechas))

    # Asegurarse que las fechas son tratadas como datetime
    fechas = pd.to_datetime(fechas)

    # Reindexar los datos para incluir todas las fechas dadas, llenando hacia adelante para obtener el precio más reciente si una fecha no es un día de trading
    datos_reindexados = datos.reindex(fechas, method='bfill')

    # Seleccionar solo la columna 'Close'
    precios_cierre = datos_reindexados['Close']

    # Calcular la variación logarítmica
    variacion_log = np.log(precios_cierre).diff().dropna()

    # Convertir a porcentaje, ajustar formato decimal y añadir símbolo de porcentaje
    variacion_log_porcentaje = variacion_log.apply(lambda x: f"{x*100:.2f}".replace('.', ',') + '%')

    # Convertir el índice a DatetimeIndex y renombrarlo
    variacion_log_porcentaje.index = pd.to_datetime(variacion_log_porcentaje.index)
    variacion_log_porcentaje.index.name = 'Date'

    # Ahora crea el DataFrame
    resultado = pd.DataFrame({
        'Variacion Logaritmica': variacion_log_porcentaje.values
    }, index=variacion_log_porcentaje.index)

    return resultado




In [20]:

def regresion(ticker):


  fechas = ["2018-01-31", "2018-02-28", "2018-03-31", "2018-04-30", "2018-05-31", "2018-06-30", "2018-07-31", "2018-08-31", "2018-09-30", "2018-10-31", "2018-11-30", "2018-12-31", "2019-01-31", "2019-02-28", "2019-03-31", "2019-04-30", "2019-05-31", "2019-06-30", "2019-07-31", "2019-08-31", "2019-09-30", "2019-10-31", "2019-11-30", "2019-12-31", "2020-01-31", "2020-02-29", "2020-03-31", "2020-04-30", "2020-05-31", "2020-06-30", "2020-07-31", "2020-08-31", "2020-09-30", "2020-10-31", "2020-11-30", "2020-12-31", "2021-01-31", "2021-02-28", "2021-03-31", "2021-04-30", "2021-05-31", "2021-06-30", "2021-07-31", "2021-08-31", "2021-09-30", "2021-10-31", "2021-11-30", "2021-12-31", "2022-01-31", "2022-02-28", "2022-03-31", "2022-04-30", "2022-05-31", "2022-06-30", "2022-07-31", "2022-08-31", "2022-09-30", "2022-10-31", "2022-11-30", "2022-12-31", "2023-01-31", "2023-02-28", "2023-03-31", "2023-04-30", "2023-05-31", "2023-06-30", "2023-07-31", "2023-08-31", "2023-09-30", "2023-10-31", "2023-11-30", "2023-12-31",]


  # Ejemplo de uso
  # Símbolo del ticker para Apple Inc.

  fechas = validar_y_convertir_fechas(fechas)


  df_variacion_log = obtener_precios_logaritmicos(ticker,fechas)
  df_variacion_log.head()




  famafrench_df = pd.read_csv('/content/general_csv_weekly_4factors.csv', sep = ';')


  # Convierte la columna de fecha a datetime
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'], format="%Y-%m-%d")

  # Si necesitas un formato específico, puedes usar el parámetro 'format'
  # df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')

  # Después de convertir, establece la columna de fecha como índice si deseas
  famafrench_df.set_index('Date', inplace=True)





  # Si ambos DataFrames tienen el índice de fecha correctamente configurado, puedes proceder directamente a merge
  df_combinado = famafrench_df.merge(df_variacion_log, left_index=True, right_index=True, how='outer')
  # Eliminar filas que contengan algún valor NaN
  df_combinado = df_combinado.dropna()
  # Asegúrate de que el índice está en formato datetime si aún no lo está
  df_combinado.index = pd.to_datetime(df_combinado.index)

  # Define el rango de fechas
  fecha_inicio = '2018-01-01'
  fecha_fin = '2023-12-31'

  # Filtra el DataFrame para incluir solo las fechas dentro del rango
  df_combinado = df_combinado[(df_combinado.index >= fecha_inicio) & (df_combinado.index <= fecha_fin)]

  # Convertir el índice de fecha a una columna regular

  df_combinado = df_combinado.round(3)

  # Asegúrate de que las columnas son tratadas como strings antes de reemplazar ',' por '.'
  df_combinado['Mkt-RF'] = df_combinado['Mkt-RF'].astype(str).str.replace(',', '.').astype(float)/100
  df_combinado['SMB'] = df_combinado['SMB'].astype(str).str.replace(',', '.').astype(float)/100
  df_combinado['HML'] = df_combinado['HML'].astype(str).str.replace(',', '.').astype(float)/100


  # Asegura que Pandas trate las columnas como strings antes de realizar operaciones de strings
  df_combinado['RF'] = df_combinado['RF'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100
  df_combinado['Variacion Logaritmica'] = df_combinado['Variacion Logaritmica'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100

  df_combinado['Fundflows'] = df_combinado['Fundflows'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100
  print(df_combinado)

  # Preparar las variables independientes
  # Añadir una constante al modelo para el término de intercepción
  X = df_combinado[['Mkt-RF', 'SMB', 'HML', 'Fundflows']]
  X = sm.add_constant(X)

  # La variable dependiente es el exceso de rendimiento de Apple
  # Asegúrate de que 'RF' y 'rendimientos apple' están en formato decimal adecuado
  Y = df_combinado['Variacion Logaritmica'] - df_combinado['RF']

  # Estimar el modelo OLS
  modelo = sm.OLS(Y, X).fit()
  print(modelo.summary())


  # Crear un DataFrame para este ticker específico con los resultados del modelo
  resultados_ticker = pd.DataFrame({
        ticker: {
            'const': modelo.params['const'],
            'pvalorconst': modelo.pvalues['const'],
            'coefSML': modelo.params['SMB'],
            'pvalorSML': modelo.pvalues['SMB'],
            'coefHML': modelo.params['HML'],
            'pvalorHML': modelo.pvalues['HML'],
            'r2': modelo.rsquared,
            'coefrmrf': modelo.params['Mkt-RF'],
            'pvalorrmrf': modelo.pvalues['Mkt-RF'],
            'coefflujos': modelo.params['Fundflows'],
            'pvalorflujos': modelo.pvalues['Fundflows'],


        }
    })

  return resultados_ticker


In [22]:
# Asume que esta es tu lista de tickers
lista_tickers = ["MSFT", "AAPL", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "BRK.B", "LLY", "AVGO", "JPM", "TSLA", "XOM", "V", "UNH", "MA", "PG", "JNJ", "HD", "MRK", "COST", "ABBV", "CRM", "CVX", "AMD", "NFLX", "BAC", "WMT", "PEP", "KO", "LIN", "TMO", "ADBE", "DIS", "ACN", "WFC", "ORCL", "CSCO", "MCD", "QCOM", "ABT", "CAT", "INTU", "AMAT", "IBM", "VZ", "GE", "CMCSA", "NOW", "INTC", "DHR", "COP", "UBER", "TXN", "PFE", "UNP", "AMGN", "PM", "LOW", "SPGI", "ISRG", "MU", "RTX", "GS", "NEE", "HON", "ETN", "AXP", "LRCX", "BKNG", "PGR", "T", "ELV", "SYK", "C", "MS", "PLD", "BLK", "MDT", "TJX", "NKE", "UPS", "SCHW", "DE", "CI", "BA", "VRTX", "BMY", "CB", "ADP", "MMC", "BSX", "REGN", "SBUX", "ADI", "LMT", "FI", "KLAC", "CVS", "BX", "MDLZ", "AMT", "SNPS", "GILD", "PANW", "CDNS", "TMUS", "CMG", "MPC", "EOG", "ICE", "TGT", "SHW", "SLB", "CME", "SO", "ZTS", "WM", "ANET", "DUK", "MO", "EQIX", "PH", "PSX", "CL", "ITW", "FCX", "PYPL", "CSX", "BDX", "MCK", "ABNB", "APH", "TT", "TDG", "USB", "GD", "ORLY", "EMR", "HCA", "NOC", "PNC", "PCAR", "AON", "FDX", "PXD", "NXPI", "MAR", "MCO", "VLO", "CEG", "CTAS", "MSI", "ROP", "ECL", "NSC", "EW", "COF", "AIG", "DXCM", "HLT", "AZO", "APD", "F", "TRV", "AJG", "ADSK", "TFC", "GM", "WELL", "MMM", "NUE", "SPG", "CPRT", "CARR", "MCHP", "URI", "ROST", "WMB", "DHI", "SMCI", "OKE", "PSA", "NEM", "OXY", "MET", "AFL", "ALL", "TEL", "GWW", "SRE", "O", "AEP", "IQV", "JCI", "AMP", "FTNT", "CCI", "MSCI", "DLR", "FAST", "FIS", "BK", "HES", "STZ", "IDXX", "KMB", "A", "DOW", "AME", "PRU", "LULU", "LEN", "MNST", "CMI", "D", "CTVA", "ODFL", "OTIS", "COR", "PAYX", "LHX", "GIS", "HUM", "CNC", "SYY", "RSG", "MLM", "CSGP", "PWR", "IR", "YUM", "EXC", "GEHC", "FANG", "IT", "HAL", "KR", "PCG", "VMC", "CTSH", "KMI", "GEV", "ACGL", "MRNA", "KVUE", "DG", "BKR", "DVN", "CDW", "EL", "ADM", "GPN", "PEG", "PPG", "VRSK", "DD", "RCL", "MPWR", "ROK", "KDP", "EA", "EFX", "EXR", "DFS", "ED", "HIG", "VICI", "FICO", "XYL", "DAL", "ANSS", "XEL", "BIIB", "FTV", "ON", "KHC", "HSY", "WST", "CBRE", "MTD", "KEYS", "WTW", "RMD", "EIX", "CHTR", "TSCO", "CAH", "WAB", "EBAY", "DLTR", "ZBH", "LYB", "TROW", "AVB", "HWM", "TRGP", "WEC", "HPQ", "WY", "NVR", "CHD", "PHM", "BLDR", "FITB", "DOV", "GLW", "RJF", "TTWO", "BR", "NDAQ", "STT", "WDC", "MTB", "HPE", "AWK", "IRM", "SBAC", "GRMN", "ALGN", "DECK", "DTE", "STLD", "ETR", "HUBB", "ULTA", "PTC", "MOH", "CPAY", "NTAP", "AXON", "EQR", "IFF", "APTV", "BAX", "GPC", "CTRA", "STE", "BALL", "ES", "ILMN", "INVH", "BRO", "PPL", "HBAN", "WAT", "FE", "ARE", "COO", "TDY", "LVS", "CBOE", "VLTO", "FSLR", "CINF", "AEE", "TXT", "MKC", "RF", "WBD", "DRI", "PFG", "J", "OMC", "NTRS", "HOLX", "IEX", "CLX", "CNP", "LH", "JBL", "WRB", "LDOS", "AVY", "EXPE", "SYF", "DPZ", "TYL", "VTR", "MAS", "ATO", "CMS", "MRO", "STX", "EXPD", "PKG", "LUV", "TSN", "FDS", "NRG", "SWKS", "VRSN", "TER", "EG", "CE", "CFG", "AKAM", "JBHT", "CCL", "ENPH", "ESS", "BBY", "SNA", "TRMB", "ALB", "BG", "EPAM", "MAA", "POOL", "CF", "ZBRA", "K", "EQT", "CAG", "SWK", "NDSN", "LYV", "DGX", "HST", "KEY", "UAL", "VTRS", "L", "LKQ", "WBA", "PNR", "DOC", "IP", "AMCR", "KMX", "RVTY", "CRL", "MGM", "ROL", "GEN", "JKHY", "WRK", "LNT", "KIM", "TAP", "AES", "EVRG", "IPG", "EMN", "SJM", "PODD", "JNPR", "ALLE", "FFIV", "HII", "UDR", "LW", "QRVO", "NI", "CPT", "TECH", "APA", "AOS", "BBWI", "MOS", "UHS", "CTLT", "INCY", "TFX", "WYNN", "HRL", "TPR", "PAYC", "NWSA", "REG", "DAY", "AIZ", "HSIC", "SOLV", "MTCH", "GL", "BF.B", "CZR", "AAL", "BXP", "CPB", "MKTX", "CHRW", "PNW", "GNRC", "BWA", "NCLH", "RHI", "ETSY", "FOXA", "BEN", "IVZ", "FMC", "FRT", "HAS", "DVA", "CMA", "BIO", "RL", "MHK",]



# DataFrame final para almacenar los resultados
df_resultados_finales = pd.DataFrame()

# Procesar cada ticker y acumular los resultados
for ticker in lista_tickers:
    try:
        resultados_ticker = regresion(ticker)
        df_resultados_finales = pd.concat([df_resultados_finales, resultados_ticker], axis=1)
    except Exception as e:
        print(f"Error al procesar {ticker}: {e}")

# Transponer el DataFrame para tener tickers como índices y resultados como columnas
df_resultados_finales = df_resultados_finales.T


# Exportar el DataFrame a CSV
df_resultados_finales.to_csv('/content/resultados_modelos.csv')

[*********************100%%**********************]  1 of 1 completed
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=11
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "
[*********************100%%**********************]  1 of 1 completed
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=11
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


            Mkt-RF     SMB     HML       RF  Fundflows  Variacion Logaritmica
Date                                                                         
2018-08-31  0.0096 -0.0003 -0.0180  0.00041    0.00683                 0.0572
2018-11-30  0.0478 -0.0181 -0.0167  0.00044    0.01956                 0.0375
2019-05-31 -0.0271 -0.0049 -0.0104  0.00052   -0.01170                -0.0544
2020-01-31 -0.0206 -0.0107 -0.0146  0.00032    0.00391                 0.0765
2020-07-31  0.0173 -0.0059 -0.0262  0.00003   -0.00794                 0.0073
2020-12-31  0.0072 -0.0218  0.0110  0.00002    0.00173                 0.0383
2021-04-30 -0.0011 -0.0070  0.0136  0.00000    0.01385                 0.0673
2021-12-31  0.0054 -0.0064  0.0142  0.00002    0.02376                 0.0172
2022-09-30 -0.0254  0.0166 -0.0081  0.00048    0.01190                -0.1157
2023-03-31  0.0356 -0.0012 -0.0014  0.00091    0.00224                 0.1449
2023-06-30  0.0236  0.0058  0.0035  0.00101    0.03236          

[*********************100%%**********************]  1 of 1 completed
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=11
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "
[*********************100%%**********************]  1 of 1 completed
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=11
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


            Mkt-RF     SMB     HML       RF  Fundflows  Variacion Logaritmica
Date                                                                         
2018-08-31  0.0096 -0.0003 -0.0180  0.00041    0.00683                 0.1365
2018-11-30  0.0478 -0.0181 -0.0167  0.00044    0.01956                -0.2547
2019-05-31 -0.0271 -0.0049 -0.0104  0.00052   -0.01170                -0.2898
2020-01-31 -0.0206 -0.0107 -0.0146  0.00032    0.00391                 0.0048
2020-07-31  0.0173 -0.0059 -0.0262  0.00003   -0.00794                 0.1112
2020-12-31  0.0072 -0.0218  0.0110  0.00002    0.00173                -0.0262
2021-04-30 -0.0011 -0.0070  0.0136  0.00000    0.01385                 0.1173
2021-12-31  0.0054 -0.0064  0.0142  0.00002    0.02376                -0.1053
2022-09-30 -0.0254  0.0166 -0.0081  0.00048    0.01190                -0.2179
2023-03-31  0.0356 -0.0012 -0.0014  0.00091    0.00224                 0.1794
2023-06-30  0.0236  0.0058  0.0035  0.00101    0.03236          

[*********************100%%**********************]  1 of 1 completed
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=11
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "
[*********************100%%**********************]  1 of 1 completed
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=11
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


...

            Mkt-RF     SMB     HML       RF  Fundflows  Variacion Logaritmica
Date                                                                         
2018-08-31  0.0096 -0.0003 -0.0180  0.00041    0.00683                 0.0056
2018-11-30  0.0478 -0.0181 -0.0167  0.00044    0.01956                -0.0296
2019-05-31 -0.0271 -0.0049 -0.0104  0.00052   -0.01170                -0.1328
2020-01-31 -0.0206 -0.0107 -0.0146  0.00032    0.00391                -0.1597
2020-07-31  0.0173 -0.0059 -0.0262  0.00003   -0.00794                 0.0110
2020-12-31  0.0072 -0.0218  0.0110  0.00002    0.00173                 0.1270
2021-04-30 -0.0011 -0.0070  0.0136  0.00000    0.01385                 0.0466
2021-12-31  0.0054 -0.0064  0.0142  0.00002    0.02376                 0.0527
2022-09-30 -0.0254  0.0166 -0.0081  0.00048    0.01190                -0.1217
2023-03-31  0.0356 -0.0012 -0.0014  0.00091    0.00224                -0.4790
2023-06-30  0.0236  0.0058  0.0035  0.00101    0.03236          

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=11
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "
[*********************100%%**********************]  1 of 1 completed
/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=11
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "
[*********************100%%**********************]  1 of 1 completed


            Mkt-RF     SMB     HML       RF  Fundflows  Variacion Logaritmica
Date                                                                         
2018-08-31  0.0096 -0.0003 -0.0180  0.00041    0.00683                -0.0162
2018-11-30  0.0478 -0.0181 -0.0167  0.00044    0.01956                -0.1514
2019-05-31 -0.0271 -0.0049 -0.0104  0.00052   -0.01170                -0.2244
2020-01-31 -0.0206 -0.0107 -0.0146  0.00032    0.00391                -0.0322
2020-07-31  0.0173 -0.0059 -0.0262  0.00003   -0.00794                -0.0170
2020-12-31  0.0072 -0.0218  0.0110  0.00002    0.00173                 0.1905
2021-04-30 -0.0011 -0.0070  0.0136  0.00000    0.01385                 0.0790
2021-12-31  0.0054 -0.0064  0.0142  0.00002    0.02376                 0.0240
2022-09-30 -0.0254  0.0166 -0.0081  0.00048    0.01190                -0.0727
2023-03-31  0.0356 -0.0012 -0.0014  0.00091    0.00224                -0.0129
2023-06-30  0.0236  0.0058  0.0035  0.00101    0.03236          

/usr/local/lib/python3.10/dist-packages/scipy/stats/_stats_py.py:1806: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=11
  warnings.warn("kurtosistest only valid for n>=20 ... continuing "


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.688
Model:                            OLS   Adj. R-squared:                  0.480
Method:                 Least Squares   F-statistic:                     3.307
Date:                Sun, 19 May 2024   Prob (F-statistic):             0.0931
Time:                        17:01:48   Log-Likelihood:                 15.021
No. Observations:                  11   AIC:                            -20.04
Df Residuals:                       6   BIC:                            -18.05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0408      0.044     -0.919      0.3

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm

def descargar_datos(ticker, start_date, end_date):
    """Descarga los datos ajustados al cierre para un único ticker y maneja errores individuales."""
    try:
        datos = yf.download(ticker, start=start_date, end=end_date)['Adj Close']
        datos.name = ticker  # Renombra la serie para evitar problemas
        return datos
    except Exception as e:
        print(f"Error descargando datos para {ticker}: {e}")
        return pd.Series()  # Devuelve una serie vacía en caso de error

def calcular_modelo(y, X):
    """Calcula el modelo de regresión para un ticker dado y devuelve los resultados."""
    try:
        X = sm.add_constant(X)  # Añadir una constante al modelo
        modelo = sm.OLS(y, X).fit()
        return {
            'Const': modelo.params['const'],
            'P-valor Const': modelo.pvalues['const'],
            'Beta': modelo.params[X.columns[1]],
            'P-valor Beta': modelo.pvalues[X.columns[1]],
            'R^2': modelo.rsquared
        }
    except Exception as e:
        print(f"Error al ajustar el modelo: {e}")
        return {}

# Define la lista de tickers y el índice del mercado
tickers = ["MSFT", "AAPL"]
market_index = "^GSPC"
start_date = "2020-01-01"
end_date = "2023-01-01"

# Descarga de datos del mercado
market_data = descargar_datos(market_index, start_date, end_date)
if market_data.empty:
    print("No se pudieron obtener datos del índice del mercado.")

resultados = pd.DataFrame()

# Procesar cada ticker individualmente
for ticker in tickers:
    ticker_data = descargar_datos(ticker, start_date, end_date)
    if not ticker_data.empty and not market_data.empty:
        log_returns_ticker = np.log(ticker_data / ticker_data.shift(1))
        log_returns_market = np.log(market_data / market_data.shift(1))
        common_index = log_returns_ticker.dropna().index.intersection(log_returns_market.dropna().index)
        y = log_returns_ticker.loc[common_index]
        X = log_returns_market.loc[common_index]
        if not y.empty and not X.empty:
            resultado = calcular_modelo(y, X)
            resultados = resultados.append(pd.DataFrame(resultado, index=[ticker]))
        else:
            print(f"No hay suficientes datos para calcular el modelo para {ticker}.")
    else:
        print(f"No se pudieron obtener datos para {ticker}.")

print(resultados)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


AttributeError: 'DataFrame' object has no attribute 'append'

In [ ]:
from datetime import datetime
import os
import shutil
import pandas as pd

pd.options.mode.chained_assignment = None  # default='warn'
pd.set_option('display.max_rows', 600)

# -*- encoding: utf-8 -*-
%matplotlib inline

# Date to use for snapshot of S&P 500 components.
snap_shot = '2018-10-23'

def get_table(filename):

    if os.path.isfile(filename):
        df = pd.read_csv(filename, index_col='date')
        return df


filename = 'S&P 500 Historical Components & Changes(04-08-2024).csv'
df = get_table(filename)
df.tail()

AttributeError: 'NoneType' object has no attribute 'tail'